# SVM – Teste (base completa, h3 + período)

SVM linear sobre a mesma base do projeto (`workspace.crimes_sp.silver_celulares_sp`), com as mesmas janelas sem vazamento e amostragem 1:1 do pipeline AutoML — usa **apenas `h3_cell` e `periodo_dia`** (one-hot). Métricas no holdout temporal (jun–ago/2026) para comparação com RandomForest e XGBoost.

**Base (24/11/2026):** este notebook usa a **base completa do estado de SP** — todas as ocorrências de roubo/furto de celular com coordenada válida, **sem recorte territorial**. Conteúdo: baseline SVM linear (C=1) no holdout temporal, fine-tuning de `C` (5-fold CV + refit do melhor) e experimento de **janelas temporais** com 5-fold CV (2026 completo, 6m e 3m vs 12m de referência) — conclusão ao final.

In [0]:
%pip install h3

In [0]:
dbutils.library.restartPython()

In [0]:
import h3
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# Configuração geral — idêntica ao pipeline do projeto (notebook AutoML)
# -----------------------------------------------------------------------------
TABLE = "workspace.crimes_sp.silver_celulares_sp"
H3_RES = 8  # células H3 de ~0.74 km² (aprox. 460 m de aresta)

COL_LAT, COL_LON = "LATITUDE", "LONGITUDE"
COL_TS = "DATAHORA_OCORRENCIA"

# Bounding box aproximada do estado de São Paulo (coordenadas válidas)
SP_LAT_MIN, SP_LAT_MAX = -25.6, -19.5
SP_LON_MIN, SP_LON_MAX = -53.2, -43.8

# Janelas temporais sem vazamento (mesmas do pipeline do projeto):
#   Histórico : ts <  2025-06-01  -> define o universo de células
#   Treino    : 2025-06-01 a 2026-06-01 (labels)
#   Holdout   : 2026-06-01 em diante (avaliação final, no tempo)
HIST_END = pd.Timestamp("2025-06-01")
SPLIT    = pd.Timestamp("2026-06-01")
MIN_CRIMES_CELL = 3               # célula entra no universo com >= 3 crimes no histórico
MAX_POS_TRAIN, MAX_POS_HOLDOUT = 60_000, 25_000
SEED = 42

print(f"Resolução H3: {H3_RES} (~{h3.average_hexagon_area(H3_RES, 'km^2'):.2f} km² por célula)")

In [0]:
# -----------------------------------------------------------------------------
# Extração: apenas ROUBO/FURTO com coordenadas válidas no estado de SP
# (PERDA/EXTRAVIO e demais rubricas ficam fora do alvo de predição),
# base COMPLETA do estado de SP (sem recorte territorial), incluindo a coluna CEP
# -----------------------------------------------------------------------------
df_spark = spark.sql(f"""
  SELECT
    CAST({COL_LAT} AS DOUBLE) AS lat,
    CAST({COL_LON} AS DOUBLE) AS lon,
    hour({COL_TS})               AS hora,
    ((dayofweek({COL_TS}) + 5) % 7) AS dia_semana,  -- 0=segunda ... 6=domingo
    month({COL_TS})               AS mes,
    {COL_TS}                      AS ts,
    NOME_MUNICIPIO_CIRC           AS municipio,
    BAIRRO                        AS bairro,
    CAST(CEP AS STRING)           AS cep
  FROM {TABLE}
  WHERE {COL_TS} IS NOT NULL
    AND {COL_LAT} BETWEEN {SP_LAT_MIN} AND {SP_LAT_MAX}
    AND {COL_LON} BETWEEN {SP_LON_MIN} AND {SP_LON_MAX}
    AND (RUBRICA LIKE 'FURTO%' OR RUBRICA LIKE 'ROUBO%')
""")

df_crimes = df_spark.toPandas()
df_crimes["ts"] = pd.to_datetime(df_crimes["ts"], utc=True).dt.tz_localize(None)

# Índice H3 de cada ocorrência (resolução H3_RES)
df_crimes["h3_cell"] = [h3.latlng_to_cell(la, lo, H3_RES) for la, lo in zip(df_crimes["lat"], df_crimes["lon"])]

print(f"Ocorrências de roubo/furto de celular — base completa (estado de SP): {len(df_crimes):,}")
print(f"CEPs distintos (não nulos): {df_crimes['cep'].nunique():,}")
print(f"Período: {df_crimes['ts'].min()} -> {df_crimes['ts'].max()}")
print(f"Células H3 distintas (res {H3_RES}): {df_crimes['h3_cell'].nunique():,}")
display(df_crimes.head(5))

In [0]:
# -----------------------------------------------------------------------------
# Dataset: mesma montagem do pipeline do projeto (mesma seed -> mesmas amostras),
# mas mantendo APENAS h3_cell e periodo_dia como features.
#
# Positivos = combinações (célula × hora × dia_semana × mês) com >= 1 crime na
# janela de labels; negativos amostrados 1:1 por rejeição apenas nos meses
# presentes na janela de labels.
# -----------------------------------------------------------------------------
def periodo_do_dia(h):
    """Derivado da hora (mesma definição do pipeline): MADRUGADA/MANHA/TARDE/NOITE."""
    if h < 6:
        return "MADRUGADA"
    if h < 12:
        return "MANHA"
    if h < 18:
        return "TARDE"
    return "NOITE"

def build_window(label_start, label_end, hist_end, max_pos, seed=SEED):
    """Monta o dataset (positivos + negativos equilibrados 1:1) de uma janela de
    labels, com universo de células definido apenas pelo histórico."""
    rng = np.random.default_rng(seed)
    hist_all = df_crimes[df_crimes["ts"] < hist_end]
    lab      = df_crimes[(df_crimes["ts"] >= label_start) & (df_crimes["ts"] < label_end)]

    cnt_cell = hist_all.groupby("h3_cell").size()
    pool = cnt_cell[cnt_cell >= MIN_CRIMES_CELL].index.to_numpy()
    lab  = lab[lab["h3_cell"].isin(pool)]

    pos_keys = lab.groupby(["h3_cell", "hora", "dia_semana", "mes"]).size().index.tolist()
    if len(pos_keys) > max_pos:
        idx = rng.choice(len(pos_keys), size=max_pos, replace=False)
        pos_keys = [pos_keys[i] for i in idx]
    pos_set = set(pos_keys)

    meses_validos = np.array(sorted(lab["mes"].unique()))

    neg_keys, seen = [], set()
    while len(neg_keys) < len(pos_keys):
        batch = max(10_000, (len(pos_keys) - len(neg_keys)) * 3)
        cand = list(zip(
            rng.choice(pool, size=batch),
            rng.integers(0, 24, size=batch),
            rng.integers(0, 7, size=batch),
            rng.choice(meses_validos, size=batch),
        ))
        for k in cand:
            if k not in pos_set and k not in seen:
                seen.add(k)
                neg_keys.append(k)
                if len(neg_keys) >= len(pos_keys):
                    break

    def assemble(keys, target):
        d = pd.DataFrame(keys, columns=["h3_cell", "hora", "dia_semana", "mes"])
        d["periodo_dia"] = d["hora"].map(periodo_do_dia)
        d["target"] = target
        return d[["h3_cell", "periodo_dia", "target"]]

    df = pd.concat([assemble(pos_keys, 1), assemble(neg_keys, 0)], ignore_index=True)
    return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

DATA_END = df_crimes["ts"].max()
train_df   = build_window(HIST_END, SPLIT, HIST_END, MAX_POS_TRAIN)
holdout_df = build_window(SPLIT, DATA_END + pd.Timedelta(days=1), SPLIT, MAX_POS_HOLDOUT)

for nome, d in [("TREINO", train_df), ("HOLDOUT", holdout_df)]:
    print(f"{nome}: {len(d):,} amostras | positivos={int(d.target.sum()):,} "
          f"({d.target.mean()*100:.1f}%) | células distintas={d.h3_cell.nunique():,}")
display(train_df.head(10))

In [0]:
# -----------------------------------------------------------------------------
# TREINO — SVM linear (LinearSVC) sobre h3_cell + periodo_dia (one-hot)
#
# h3_cell e periodo_dia são categóricos: são codificados em one-hot DENTRO do
# pipeline (treino fit apenas na base de treino, sem vazamento). O SVM linear
# escala para as 120k amostras; a função de decisão serve de score para AUC.
# -----------------------------------------------------------------------------
import time
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

FEATURES = ["h3_cell", "periodo_dia"]
C_SVM = 1.0

X_train, y_train = train_df[FEATURES], train_df["target"]
X_test,  y_test  = holdout_df[FEATURES], holdout_df["target"]

svm_model = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ("clf", LinearSVC(C=C_SVM, random_state=SEED, dual="auto")),
])

t0 = time.time()
svm_model.fit(X_train, y_train)
tempo_treino = time.time() - t0

y_pred  = svm_model.predict(X_test)
y_score = svm_model.decision_function(X_test)

metrics = {
    "acuracia":  accuracy_score(y_test, y_pred),
    "precisao":  precision_score(y_test, y_pred),
    "recall":    recall_score(y_test, y_pred),
    "f1":        f1_score(y_test, y_pred),
    "auc":       roc_auc_score(y_test, y_score),
}

print(f"Tempo de treino: {tempo_treino:.1f}s | amostras: treino={len(X_train):,}, holdout={len(X_test):,}")
print("\n### Métricas no holdout (jun–ago/2026) — SVM linear, features: h3_cell + periodo_dia")
for k, v in metrics.items():
    print(f"  {k:<10}: {v:.4f}")

print("\n### Matriz de confusão (linhas = real, colunas = previsto)")
cm = confusion_matrix(y_test, y_pred)
display(pd.DataFrame(cm, index=["real 0", "real 1"], columns=["prev 0", "prev 1"]))

print("\n### Classification report")
print(classification_report(y_test, y_pred, target_names=["sem crime", "crime"], digits=4))

# Comparação com os modelos já testados no projeto (mesmo holdout)
print("### Comparação com referências do projeto (mesmas janelas/holdout):")
comparacao = pd.DataFrame([
    {"modelo": "RandomForest (24 features, registrado)", "f1": 0.831, "auc": 0.906},
    {"modelo": "XGBoost base pura (8 features)",        "f1": 0.805, "auc": 0.886},
    {"modelo": "XGBoost 4 melhores features",          "f1": 0.785, "auc": 0.869},
    {"modelo": "SVM linear (h3_cell + periodo_dia)",    "f1": round(metrics["f1"], 4), "auc": round(metrics["auc"], 4)},
])
display(comparacao)

In [0]:
# -----------------------------------------------------------------------------
# FINE-TUNING — varredura do hiperparâmetro C do SVM linear
#
# A separação temporal já está garantida (treino 2025-06→2026-06, holdout depois
# e intocado até o final). Dentro do treino, as amostras são independentes entre
# si (combinações célula × período), então a validação usa split estratificado
# com 5-fold CV — mais robusto que um único split.
#
# Ao final: refit do melhor C na base de treino COMPLETA e avaliação no holdout,
# comparando com o baseline C=1.0.
# -----------------------------------------------------------------------------
import time
from sklearn.model_selection import StratifiedKFold, cross_validate

Cs = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

resultados = []
for c in Cs:
    pipe = Pipeline([
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ("clf", LinearSVC(C=c, random_state=SEED, dual="auto")),
    ])
    cvres = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring={"f1": "f1", "auc": "roc_auc"}, n_jobs=-1,
    )
    resultados.append({
        "C": c,
        "f1_medio": cvres["test_f1"].mean(),
        "f1_std": cvres["test_f1"].std(),
        "auc_medio": cvres["test_auc"].mean(),
        "tempo_s": cvres["fit_time"].mean(),
    })

tab_cv = pd.DataFrame(resultados)
print("### Varredura de C — 5-fold CV estratificado na base de treino deste run")
display(tab_cv.round(4))

melhor = tab_cv.loc[tab_cv["f1_medio"].idxmax()]
print(f"\nMelhor C pelo F1 médio da validação: C={melhor['C']} "
      f"(F1 {melhor['f1_medio']:.4f} ± {melhor['f1_std']:.4f} | AUC {melhor['auc_medio']:.4f})")

# ---- refit do melhor C no treino completo + holdout ----
svm_tuned = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ("clf", LinearSVC(C=float(melhor["C"]), random_state=SEED, dual="auto")),
])
t0 = time.time()
svm_tuned.fit(X_train, y_train)
tempo_tuned = time.time() - t0

y_pred_t  = svm_tuned.predict(X_test)
y_score_t = svm_tuned.decision_function(X_test)
metrics_tuned = {
    "acuracia": accuracy_score(y_test, y_pred_t),
    "precisao": precision_score(y_test, y_pred_t),
    "recall":   recall_score(y_test, y_pred_t),
    "f1":       f1_score(y_test, y_pred_t),
    "auc":      roc_auc_score(y_test, y_score_t),
}

print(f"\n### Holdout — SVM linear com C={melhor['C']} (refit no treino completo, {tempo_tuned:.1f}s)")
for k, v in metrics_tuned.items():
    print(f"  {k:<10}: {v:.4f}")

cm_t = confusion_matrix(y_test, y_pred_t)
display(pd.DataFrame(cm_t, index=["real 0", "real 1"], columns=["prev 0", "prev 1"]))

print("### Fine-tuning vs baseline (mesmo holdout)")
display(pd.DataFrame([
    {"modelo": f"SVM linear C=1.0 (baseline)", "f1": round(metrics["f1"], 4), "auc": round(metrics["auc"], 4)},
    {"modelo": f"SVM linear C={melhor['C']} (tunado)", "f1": round(metrics_tuned["f1"], 4), "auc": round(metrics_tuned["auc"], 4)},
]))

In [0]:
# -----------------------------------------------------------------------------
# MLFLOW — registro do modelo com melhores métricas no holdout
#
# O SVM tunado (refit do melhor C da varredura, célula anterior) supera o
# baseline C=1: F1 0.7826 / AUC 0.8514 vs 0.7819 / 0.8503. Assinatura e input
# example incluídos para compatibilidade com Unity Catalog / Model Serving.
# -----------------------------------------------------------------------------
import mlflow
from mlflow.models import infer_signature

signature = infer_signature(X_test.head(10), svm_tuned.predict(X_test.head(10)))

mlflow.set_experiment("/Users/felipek0606@gmail.com/svm-crime-risk-celulares-sp")

with mlflow.start_run(run_name=f"svm_linear_h3_periodo_C{melhor['C']}"):
    mlflow.log_params({
        "modelo": "LinearSVC + OneHotEncoder (pipeline sklearn)",
        "features": "h3_cell, periodo_dia",
        "C": float(melhor["C"]),
        "base": "workspace.crimes_sp.silver_celulares_sp (completa, estado de SP)",
        "janela_treino": "2025-06-01 a 2026-06-01",
        "janela_holdout": "2026-06-01 a 2026-08-31",
        "amostras_treino": len(X_train),
        "amostras_holdout": len(X_test),
        "seed": SEED,
        "tempo_refit_s": round(tempo_tuned, 1),
    })
    mlflow.log_metrics(metrics_tuned)
    model_info = mlflow.sklearn.log_model(
        svm_tuned,
        name="svm_linear_h3_periodo",
        signature=signature,
        input_example=X_test.head(5),
    )
    print(f"Run: {mlflow.active_run().info.run_id}")
    print(f"Modelo logado no MLflow: {model_info.model_uri}")

## Janelas temporais + k-fold — SVM de 2 features (h3_cell + periodo_dia)

Mesmo gerador de datasets do notebook (positivos = combinações célula × hora × dia × mês com crime na janela; negativos 1:1 por rejeição nos meses da própria janela; universo = células com ≥ 3 crimes **antes** da janela, sem vazamento), variando o recorte temporal:

* **Ano 2026 completo** (jan/2026 → fim dos dados)
* **Últimos 6 meses** (mar/2026 → ago/2026)
* **Últimos 3 meses** (jun/2026 → ago/2026)
* **Referência: 12 meses** (jun/2025 → jun/2026, janela do projeto)

Todos os positivos de cada janela (sem cap) e **5-fold CV estratificado** (F1, AUC, precisão, recall, acurácia fold a fold + média ± desvio).

**Atenção na leitura:** o k-fold mede a qualidade *dentro* de cada janela (consistência do modelo naquele recorte), não substitui o holdout temporal — janelas menores têm menos amostras, então queda de métrica pode vir de menos dados, não só de dados menos recentes.

In [0]:
# -----------------------------------------------------------------------------
# JANELAS TEMPORAIS + K-FOLD — SVM linear, apenas h3_cell + periodo_dia
#
# Variando o recorte temporal (2026 completo, últimos 6 meses, últimos 3 meses
# e a janela de 12 meses do projeto como referência), TODOS os positivos de
# cada janela + negativos 1:1. Qualidade acompanhada por 5-fold CV
# estratificado: F1, AUC, precisão, recall e acurácia fold a fold.
# -----------------------------------------------------------------------------
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

# ---- constantes do pipeline (idem às células de configuração; redefinidas
#      aqui para a célula sobreviver a restart de kernel)
SEED, MIN_CRIMES_CELL = 42, 3

# ---- garante df_crimes no kernel (extração idêntica às células anteriores)
try:
    df_crimes
except NameError:
    import h3
    df_crimes = spark.sql("""
      SELECT CAST(LATITUDE AS DOUBLE) AS lat, CAST(LONGITUDE AS DOUBLE) AS lon,
             hour(DATAHORA_OCORRENCIA) AS hora,
             ((dayofweek(DATAHORA_OCORRENCIA)+5)%7) AS dia_semana,
             month(DATAHORA_OCORRENCIA) AS mes,
             DATAHORA_OCORRENCIA AS ts
      FROM workspace.crimes_sp.silver_celulares_sp
      WHERE DATAHORA_OCORRENCIA IS NOT NULL
        AND LATITUDE BETWEEN -25.6 AND -19.5
        AND LONGITUDE BETWEEN -53.2 AND -43.8
        AND (RUBRICA LIKE 'FURTO%' OR RUBRICA LIKE 'ROUBO%')  -- base completa (idem extração)
    """).toPandas()
    df_crimes["ts"] = pd.to_datetime(df_crimes["ts"], utc=True).dt.tz_localize(None)
    df_crimes["h3_cell"] = [h3.latlng_to_cell(a, b, 8) for a, b in zip(df_crimes["lat"], df_crimes["lon"])]

DATA_END_W = df_crimes["ts"].max()
FIM_W      = DATA_END_W + pd.Timedelta(days=1)

def periodo_do_dia(h):
    if h < 6:  return "MADRUGADA"
    if h < 12: return "MANHA"
    if h < 18: return "TARDE"
    return "NOITE"

def build_window_cv(label_start, label_end, seed=SEED):
    """Todos os positivos da janela + negativos 1:1; universo de células pelo
    histórico ANTERIOR à janela (>= 3 crimes) — mesma régua do pipeline."""
    rng = np.random.default_rng(seed)
    hist = df_crimes[df_crimes["ts"] < label_start]
    lab  = df_crimes[(df_crimes["ts"] >= label_start) & (df_crimes["ts"] < label_end)]
    cnt  = hist.groupby("h3_cell").size()
    pool = cnt[cnt >= MIN_CRIMES_CELL].index.to_numpy()
    lab  = lab[lab["h3_cell"].isin(pool)]

    pos_keys = lab.groupby(["h3_cell", "hora", "dia_semana", "mes"]).size().index.tolist()
    pos_set  = set(pos_keys)
    meses_validos = np.array(sorted(lab["mes"].unique()))

    neg_keys, seen = [], set()
    while len(neg_keys) < len(pos_keys):
        batch = max(10_000, (len(pos_keys) - len(neg_keys)) * 3)
        cand = list(zip(rng.choice(pool, size=batch),
                        rng.integers(0, 24, size=batch),
                        rng.integers(0, 7, size=batch),
                        rng.choice(meses_validos, size=batch)))
        for k in cand:
            if k not in pos_set and k not in seen:
                seen.add(k); neg_keys.append(k)
                if len(neg_keys) >= len(pos_keys):
                    break

    def assemble(keys, target):
        d = pd.DataFrame(keys, columns=["h3_cell", "hora", "dia_semana", "mes"])
        d["periodo_dia"] = d["hora"].map(periodo_do_dia)
        d["target"] = target
        return d[["h3_cell", "periodo_dia", "target"]]

    df = pd.concat([assemble(pos_keys, 1), assemble(neg_keys, 0)], ignore_index=True)
    return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

JANELAS = [
    ("Ano 2026 completo (jan–ago/26)",       pd.Timestamp("2026-01-01"), FIM_W),
    ("Últimos 6 meses (mar–ago/26)",         FIM_W - pd.DateOffset(months=6), FIM_W),
    ("Últimos 3 meses (jun–ago/26)",         FIM_W - pd.DateOffset(months=3), FIM_W),
    ("Referência 12 meses (jun/25–jun/26)",  pd.Timestamp("2025-06-01"), pd.Timestamp("2026-06-01")),
]

FEATURES = ["h3_cell", "periodo_dia"]
def make_svm():
    return Pipeline([
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ("clf", LinearSVC(C=1.0, random_state=SEED, dual="auto")),
    ])

resumo_folds, tamanhos, t0 = [], {}, time.time()
for nome, ls, le in JANELAS:
    d = build_window_cv(ls, le)
    X, y = d[FEATURES], d["target"].to_numpy()
    tamanhos[nome] = (len(d), int(y.sum()), d["h3_cell"].nunique())
    print(f"[{time.time()-t0:.0f}s] {nome}: {len(d):,} amostras | positivos={int(y.sum()):,} "
          f"| células distintas={d['h3_cell'].nunique():,}")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    for fold, (tr, te) in enumerate(skf.split(X, y), 1):
        tf = time.time()
        pipe  = make_svm().fit(X.iloc[tr], y[tr])
        score = pipe.decision_function(X.iloc[te])
        pred  = (score >= 0).astype(int)
        yt = y[te]
        resumo_folds.append({
            "janela": nome, "fold": fold, "tempo_s": round(time.time() - tf, 1),
            "acuracia": round(accuracy_score(yt, pred), 4),
            "precisao": round(precision_score(yt, pred), 4),
            "recall":   round(recall_score(yt, pred), 4),
            "f1":       round(f1_score(yt, pred), 4),
            "auc":      round(roc_auc_score(yt, score), 4),
        })
    print(f"[{time.time()-t0:.0f}s] CV de '{nome}' concluída")

df_folds = pd.DataFrame(resumo_folds)

linhas = []
for nome, _, _ in JANELAS:
    g = df_folds[df_folds["janela"] == nome]
    linhas.append({
        "janela": nome,
        "amostras": f"{tamanhos[nome][0]:,}",
        "f1_medio": round(g.f1.mean(), 4), "f1_std": round(g.f1.std(), 4),
        "auc_medio": round(g.auc.mean(), 4), "auc_std": round(g.auc.std(), 4),
        "precisao": round(g.precisao.mean(), 4),
        "recall":   round(g.recall.mean(), 4),
        "acuracia": round(g.acuracia.mean(), 4),
    })

print(f"\n### SVM linear (h3_cell + periodo_dia, C=1) — 5-fold CV por janela temporal ({time.time()-t0:.0f}s)")
display(pd.DataFrame(linhas))

print("\n### Métricas fold a fold")
display(df_folds)

## Janelas temporais + 5-fold CV — SVM de 2 features na base completa (24/11/2026)

SVM linear (C=1, apenas `h3_cell` + `periodo_dia`), todos os positivos da janela + negativos 1:1, universo sem vazamento (células com ≥ 3 crimes **antes** da janela), 5-fold CV estratificado — na **base completa do estado de SP** (refeitura da execução de 19/11, que rodava na amostra top-20 + CENTRO):

| Janela | Amostras | Células | F1 (médio ± std) | AUC (médio ± std) |
| --- | --- | --- | --- | --- |
| Ano 2026 completo (jan–ago/26) | 220.730 | 7.643 | 0,7829 ± 0,0021 | 0,8576 ± 0,0012 |
| Últimos 6 meses (mar–ago/26) | 169.212 | 8.040 | **0,7881 ± 0,0025** | **0,8605 ± 0,0025** |
| Últimos 3 meses (jun–ago/26) | 79.962 | 8.573 | 0,7878 ± 0,0040 | 0,8587 ± 0,0028 |
| Referência 12 meses (jun/25–jun/26) | 330.388 | 5.469 | 0,7566 ± 0,0015 | 0,8326 ± 0,0014 |

**Leitura:**

* **Recência vale mais que volume** — a janela de 12 meses tem 4× as amostras da de 3 meses e entrega F1 ~3 p.p. atrás (0,7566 vs 0,7878; AUC 0,8326 vs ~0,86): dados de 2025 diluem o padrão recente de concentração célula × período.
* **Retornos decrescentes abaixo de 6 meses** — 3 meses ≈ 6 meses (0,7878 vs 0,7881) com menos da metade das amostras; os meses 4–6 atrás praticamente não acrescentam informação.
* **Diferenças reais, não ruído** — desvios entre folds pequenos (≤ 0,004) contra diferenças de ~3 p.p. entre janelas.
* **Padrão generalizável** — o mesmo efeito aparecia na execução anterior (top-20 + CENTRO: 12m 0,8021 → 6m 0,8436): a recência ajuda em qualquer território, não é artefato da amostra.

**Caveats:** o k-fold mede consistência **dentro** da janela, não generalização para o futuro; as janelas de 6m/3m se sobrepõem ao período do holdout (jun–ago/26), então o holdout atual não as valida. Confirmação rigorosa: treinar janelas **anteriores** a jun/2026 (ex.: dez/25–jun/26 e mar–jun/26) e testar todas no mesmo holdout temporal.

**Implicação prática:** treinar numa **janela móvel de ~6 meses** e re-treinar com frequência — modelo mais fresco, mais barato (menos amostras) e de qualidade igual ou superior à acumulação longa.
